# Cross-Dataset Testing

This notebook evaluates the KMU-FED trained SqueezeNet 1.1 model on unseen datasets.

The model was trained only on KMU-FED.

In this notebook, the same saved model is tested on:

- KDEF
- FER2013

No retraining or fine-tuning is performed.

In [ ]:
from google.colab import drive
from pathlib import Path
import random

import pandas as pd
import numpy as np

from PIL import Image, ImageOps

import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

In [ ]:
drive.mount("/content/gdrive")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)

## Project Paths

This section defines paths to the saved KMU-FED model, cropped manifests, and cross-dataset result folder.

In [ ]:
PROJECT_DIR = Path("/content/gdrive/MyDrive/FER_Generalization_Project")

CROPPED_MANIFEST_DIR = PROJECT_DIR / "data_processed" / "manifests_cropped"

MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results" / "cross_dataset_testing"

BEST_MODEL_PATH = MODEL_DIR / "squeezenet_kmu_fed_best.pth"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Project exists:", PROJECT_DIR.exists())

print("\nCropped manifest directory:", CROPPED_MANIFEST_DIR)
print("Cropped manifest directory exists:", CROPPED_MANIFEST_DIR.exists())

print("\nBest model path:", BEST_MODEL_PATH)
print("Best model exists:", BEST_MODEL_PATH.exists())

print("\nResults directory:", RESULTS_DIR)

In [ ]:
print("Cropped manifest files:")

if CROPPED_MANIFEST_DIR.exists():
    for item in CROPPED_MANIFEST_DIR.iterdir():
        print(item.name)
else:
    print("Cropped manifest directory not found.")

## Load Cropped Test Manifests

This section loads the KMU-FED, KDEF, and FER2013 cropped test manifests.

KMU-FED is included for comparison with the cross-dataset results.

In [ ]:
kmu_test_df = pd.read_csv(CROPPED_MANIFEST_DIR / "kmu_test_cropped.csv")
kdef_test_df = pd.read_csv(CROPPED_MANIFEST_DIR / "kdef_frontal_test_cropped.csv")
fer2013_test_df = pd.read_csv(CROPPED_MANIFEST_DIR / "fer2013_test_cropped.csv")

print("KMU-FED test:", kmu_test_df.shape)
print("KDEF test:", kdef_test_df.shape)
print("FER2013 test:", fer2013_test_df.shape)

display(kdef_test_df.head())

In [ ]:
def fix_drive_paths(df):
    """
    Updates saved Colab paths from /content/drive/MyDrive to /content/gdrive/MyDrive.
    This is needed because preprocessing may have saved paths using /content/drive.
    """
    old_prefix = "/content/drive/MyDrive"
    new_prefix = "/content/gdrive/MyDrive"

    path_columns = ["path", "original_path", "cropped_path"]

    for col in path_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(
                old_prefix,
                new_prefix,
                regex=False
            )

    return df


kmu_test_df = fix_drive_paths(kmu_test_df)
kdef_test_df = fix_drive_paths(kdef_test_df)
fer2013_test_df = fix_drive_paths(fer2013_test_df)

print("Paths fixed.")

In [ ]:
def check_image_paths(df, dataset_name, path_col="cropped_path"):
    existing_count = df[path_col].apply(lambda p: Path(p).exists()).sum()
    total_count = len(df)

    print(f"{dataset_name}: {existing_count}/{total_count} images found")

    if existing_count < total_count:
        missing_df = df[~df[path_col].apply(lambda p: Path(p).exists())]
        print("\nExample missing paths:")
        display(missing_df[[path_col, "label"]].head())


check_image_paths(kmu_test_df, "KMU-FED test")
check_image_paths(kdef_test_df, "KDEF test")
check_image_paths(fer2013_test_df, "FER2013 test")

In [ ]:
print("KMU-FED test label counts:")
print(kmu_test_df["label"].value_counts())

print("\nKDEF test label counts:")
print(kdef_test_df["label"].value_counts())

print("\nFER2013 test label counts:")
print(fer2013_test_df["label"].value_counts())

## Define Classes and Transform

The same six emotion classes and ImageNet normalization are used as in the KMU-FED training notebook.

In [ ]:
class_names = [
    "anger",
    "disgust",
    "fear",
    "happy",
    "sadness",
    "surprise"
]

num_classes = len(class_names)

label_to_idx = {label: idx for idx, label in enumerate(class_names)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

print("Number of classes:", num_classes)
print("Label mapping:", label_to_idx)

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## Dataset and DataLoaders

This section creates PyTorch datasets and dataloaders for KMU-FED, KDEF, and FER2013 test sets.

In [ ]:
class FERManifestDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["cropped_path"]
        label = int(row["label_idx"])

        image = Image.open(image_path)
        image = ImageOps.exif_transpose(image)
        image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.long)

        return image, label

In [ ]:
batch_size = 32

kmu_test_dataset = FERManifestDataset(kmu_test_df, transform=test_transform)
kdef_test_dataset = FERManifestDataset(kdef_test_df, transform=test_transform)
fer2013_test_dataset = FERManifestDataset(fer2013_test_df, transform=test_transform)

kmu_test_loader = DataLoader(
    kmu_test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

kdef_test_loader = DataLoader(
    kdef_test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

fer2013_test_loader = DataLoader(
    fer2013_test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

print("KMU-FED test batches:", len(kmu_test_loader))
print("KDEF test batches:", len(kdef_test_loader))
print("FER2013 test batches:", len(fer2013_test_loader))

In [ ]:
images, labels = next(iter(kdef_test_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

## Load Trained KMU-FED Model

This section recreates the SqueezeNet 1.1 architecture and loads the saved KMU-FED trained weights.

The model is not retrained.

In [ ]:
def create_squeezenet_model(num_classes):
    """
    Creates SqueezeNet 1.1 architecture and modifies the final classifier
    for six emotion classes.
    """

    model = models.squeezenet1_1(weights=None)

    model.classifier[1] = nn.Conv2d(
        in_channels=512,
        out_channels=num_classes,
        kernel_size=(1, 1),
        stride=(1, 1)
    )

    model.num_classes = num_classes

    return model

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

model = create_squeezenet_model(num_classes)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)

model.eval()

print("Loaded model from:", BEST_MODEL_PATH)
print("Saved best KMU-FED test accuracy:", checkpoint["best_test_accuracy"])

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
def evaluate_model(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)

            correct_predictions += (preds == labels).sum().item()
            total_samples += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss = running_loss / total_samples
    test_accuracy = correct_predictions / total_samples

    return test_loss, test_accuracy, all_preds, all_labels

In [ ]:
kdef_loss, kdef_accuracy, kdef_preds, kdef_labels = evaluate_model(
    model=model,
    dataloader=kdef_test_loader,
    criterion=criterion,
    device=device
)

print("KDEF Test Loss:", round(kdef_loss, 4))
print("KDEF Test Accuracy:", round(kdef_accuracy, 4))

In [ ]:
kdef_report = classification_report(
    kdef_labels,
    kdef_preds,
    target_names=class_names,
    digits=4
)

print(kdef_report)

kdef_report_path = RESULTS_DIR / "kdef_classification_report.txt"

with open(kdef_report_path, "w") as f:
    f.write(kdef_report)

print("KDEF classification report saved to:", kdef_report_path)

In [ ]:
kdef_cm = confusion_matrix(kdef_labels, kdef_preds)

plt.figure(figsize=(8, 6))
plt.imshow(kdef_cm)
plt.title("KDEF Confusion Matrix - KMU-FED Trained Model")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(ticks=np.arange(num_classes), labels=class_names, rotation=45)
plt.yticks(ticks=np.arange(num_classes), labels=class_names)

for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, kdef_cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
kdef_cm_df = pd.DataFrame(
    kdef_cm,
    index=[f"true_{label}" for label in class_names],
    columns=[f"pred_{label}" for label in class_names]
)

kdef_cm_path = RESULTS_DIR / "kdef_confusion_matrix.csv"
kdef_cm_df.to_csv(kdef_cm_path)

display(kdef_cm_df)

print("KDEF confusion matrix saved to:", kdef_cm_path)

In [ ]:
kdef_precision, kdef_recall, kdef_f1, kdef_support = precision_recall_fscore_support(
    kdef_labels,
    kdef_preds,
    labels=list(range(num_classes)),
    zero_division=0
)

kdef_class_accuracy = kdef_cm.diagonal() / kdef_cm.sum(axis=1)

kdef_class_metrics_df = pd.DataFrame({
    "Class": class_names,
    "Accuracy": kdef_class_accuracy,
    "Precision": kdef_precision,
    "Recall": kdef_recall,
    "F1-score": kdef_f1,
    "Support": kdef_support
})

display(kdef_class_metrics_df)

kdef_class_metrics_path = RESULTS_DIR / "kdef_classwise_metrics.csv"
kdef_class_metrics_df.to_csv(kdef_class_metrics_path, index=False)

print("KDEF class-wise metrics saved to:", kdef_class_metrics_path)

In [ ]:
predicted_labels = [class_names[i] for i in kdef_preds]
true_labels = [class_names[i] for i in kdef_labels]

prediction_distribution_df = pd.DataFrame({
    "Predicted Label": predicted_labels
})

prediction_counts = prediction_distribution_df["Predicted Label"].value_counts().reindex(class_names, fill_value=0)

print("KDEF prediction distribution:")
print(prediction_counts)

prediction_counts_path = RESULTS_DIR / "kdef_prediction_distribution.csv"
prediction_counts.to_csv(prediction_counts_path)

print("KDEF prediction distribution saved to:", prediction_counts_path)

In [ ]:
kdef_prf_plot_path = RESULTS_DIR / "kdef_precision_recall_f1.png"

x = np.arange(len(class_names))
width = 0.25

plt.figure(figsize=(10, 6))

plt.bar(x - width, kdef_class_metrics_df["Precision"], width, label="Precision")
plt.bar(x, kdef_class_metrics_df["Recall"], width, label="Recall")
plt.bar(x + width, kdef_class_metrics_df["F1-score"], width, label="F1-score")

plt.xticks(x, class_names, rotation=45)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("KDEF Class-wise Precision, Recall, and F1-score")
plt.legend()

plt.tight_layout()
plt.savefig(kdef_prf_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("KDEF precision-recall-F1 plot saved to:", kdef_prf_plot_path)

In [ ]:
kmu_baseline_accuracy = checkpoint["best_test_accuracy"]

kdef_summary = {
    "model": "SqueezeNet 1.1",
    "train_dataset": "KMU-FED cropped train",
    "test_dataset": "KDEF cropped frontal test",
    "retraining": "No",
    "fine_tuning": "No",
    "num_classes": num_classes,
    "batch_size": batch_size,
    "kmu_fed_baseline_accuracy": kmu_baseline_accuracy,
    "kdef_test_accuracy": kdef_accuracy,
    "kdef_test_loss": kdef_loss,
    "accuracy_drop_from_kmu_to_kdef": kmu_baseline_accuracy - kdef_accuracy,
    "model_path": str(BEST_MODEL_PATH)
}

kdef_summary_df = pd.DataFrame([kdef_summary])

kdef_summary_path = RESULTS_DIR / "kdef_cross_dataset_summary.csv"
kdef_summary_df.to_csv(kdef_summary_path, index=False)

display(kdef_summary_df)

print("KDEF summary saved to:", kdef_summary_path)

In [ ]:
kdef_prediction_analysis_df = pd.DataFrame({
    "True Label": [class_names[i] for i in kdef_labels],
    "Predicted Label": [class_names[i] for i in kdef_preds]
})

disgust_analysis = kdef_prediction_analysis_df[
    kdef_prediction_analysis_df["True Label"] == "disgust"
]

print("Number of true disgust images:", len(disgust_analysis))

print("\nTrue disgust images were predicted as:")
print(disgust_analysis["Predicted Label"].value_counts())